# Preprocesamiento del texto

Una vez realizado el análisis exploratorio del conjunto de datos, la siguiente etapa consiste en preparar la información para el entrenamiento de los modelos de Procesamiento del Lenguaje Natural (NLP). El objetivo principal es transformar las reseñas en una representación más limpia, consistente y adecuada para las técnicas de extracción de características y clasificación.

Durante este proceso se realizarán tareas de limpieza estructural del conjunto de datos, tratamiento de valores faltantes, normalización del texto, eliminación de elementos irrelevantes y lematización de las palabras. Estas transformaciones permitirán reducir el ruido presente en las reseñas y mejorar la calidad de las representaciones utilizadas posteriormente por los modelos de aprendizaje automático.

Finalmente, el conjunto de datos preprocesado será almacenado para su utilización en las etapas de ingeniería de características, entrenamiento y evaluación de los modelos.

## Importación de librerías

En esta sección se importan las librerías necesarias para la manipulación de datos, el procesamiento del lenguaje natural y la visualización de resultados. Asimismo, se configura el entorno de trabajo para garantizar la reproducibilidad del análisis.

In [90]:
# ===================================
# Manipulación de datos
# ===================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# ===================================
# Visualización
# ===================================

import matplotlib.pyplot as plt
import seaborn as sns

# ===================================
# Procesamiento de texto
# ===================================

import nltk

# ===================================
# Configuración
# ===================================

import warnings

warnings.filterwarnings("ignore")

plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

## Carga del dataset

El conjunto de datos se carga mediante la función `load_data()` implementada en el módulo `src.data.load_data`, manteniendo una estructura modular y reutilizable. Una vez importado, se verifica su correcta lectura antes de comenzar las tareas de preprocesamiento.

In [91]:
# Agregar la raíz del proyecto al PATH
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

# Importar función de carga
from src.data.load_data import load_data

# Cargar dataset
df = load_data()

print(f"Dimensiones del dataset: {df.shape}")

display(df.head())

Dimensiones del dataset: (23486, 11)


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i n...",5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and really wanted it to work for me. i initially ordered th...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get no...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length t...,5,1,6,General,Tops,Blouses


## Limpieza inicial del dataset

Antes de aplicar las técnicas de procesamiento del lenguaje natural, resulta necesario realizar una limpieza inicial del conjunto de datos. En esta etapa se eliminarán columnas sin utilidad para el problema de clasificación, se tratarán los valores faltantes más relevantes y se preparará el texto que será utilizado posteriormente durante el preprocesamiento.

Estas tareas permiten obtener un conjunto de datos consistente, reduciendo la presencia de información redundante o incompleta antes de iniciar la normalización del texto.

In [92]:
print("Antes:", len(df))

df = df.dropna(subset=["Review Text"])

df["Title"] = df["Title"].fillna("")

print("Después:", len(df))

Antes: 23486
Después: 22641


In [93]:
df["Text"] = (
    df["Title"]
    + " "
    + df["Review Text"]
)

display(
    df[
        [
            "Title",
            "Review Text",
            "Text"
        ]
    ].head()
)

,Title,Review Text,Text
0,,Absolutely wonderful - silky and sexy and comfortable,Absolutely wonderful - silky and sexy and comfortable
1,,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i n...","Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i ..."
2,Some major design flaws,I had such high hopes for this dress and really wanted it to work for me. i initially ordered th...,Some major design flaws I had such high hopes for this dress and really wanted it to work for me...
3,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get no...","My favorite buy! I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i ..."
4,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length t...,Flattering shirt This shirt is very flattering to all due to the adjustable front tie. it is the...


In [94]:
columns = [
    "Text",
    "Rating",
    "Age",
    "Recommended IND",
    "Positive Feedback Count",
    "Division Name",
    "Department Name",
    "Class Name"
]

df = df[columns]

print(df.shape)
display(df.head())

(22641, 8)


,Text,Rating,Age,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,Absolutely wonderful - silky and sexy and comfortable,4,33,1,0,Initmates,Intimate,Intimates
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i ...",5,34,1,4,General,Dresses,Dresses
2,Some major design flaws I had such high hopes for this dress and really wanted it to work for me...,3,60,0,0,General,Dresses,Dresses
3,"My favorite buy! I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i ...",5,50,1,0,General Petite,Bottoms,Pants
4,Flattering shirt This shirt is very flattering to all due to the adjustable front tie. it is the...,5,47,1,6,General,Tops,Blouses


### Conclusiones

Tras la limpieza inicial se eliminó la columna **Unnamed: 0**, correspondiente al índice original del archivo, y se descartaron únicamente aquellas observaciones que no contenían información en la variable **Review Text**, dado que constituyen registros inutilizables para una tarea de clasificación basada en texto.

Asimismo, los valores faltantes de **Title** fueron reemplazados por una cadena vacía y se construyó una nueva variable denominada **Text**, resultado de concatenar el título y el contenido de cada reseña. Esta transformación permite disponer de una única representación textual que será utilizada durante todo el pipeline de preprocesamiento y entrenamiento de los modelos.

## Preprocesamiento del texto

Una vez realizada la limpieza inicial del conjunto de datos, se procede al preprocesamiento del texto. El objetivo de esta etapa es transformar las reseñas en una representación más uniforme, reduciendo el ruido y conservando únicamente la información relevante para el problema de clasificación.

Para ello se implementará un pipeline de preprocesamiento que incluirá tareas de normalización, eliminación de caracteres innecesarios, tokenización, eliminación de palabras vacías (stopwords) y lematización. Todas estas transformaciones serán implementadas de forma modular para facilitar su reutilización durante las etapas de entrenamiento e inferencia.

In [95]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lauta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lauta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lauta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lauta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\lauta\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [96]:
from src.data import preprocessing as prep

In [97]:
def show_step(function, data):
    print("=" * 80)
    print(f"Paso: {function.__name__}")
    print("=" * 80)

    print("\nEntrada:")
    print(data)

    result = function(data)

    print("\nSalida:")
    print(result)
    print()

    return result

example = df["Text"].sample(1, random_state=RANDOM_STATE).iloc[0]

text = example
print(text)

Stunning! This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you won't regret it!

i'm normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.


In [98]:
text = show_step(prep.expand_contractions, text)

Paso: expand_contractions

Entrada:
Stunning! This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you won't regret it!

i'm normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

Salida:
Stunning! This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.



In [99]:
text = show_step(prep.normalize_text, text)

Paso: normalize_text

Entrada:
Stunning! This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

Salida:
stunning! this sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.



In [100]:
show_step(prep.remove_punctuation, example)

Paso: remove_punctuation

Entrada:
Stunning! This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you won't regret it!

i'm normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

Salida:
Stunning This sweater is so beautiful on it is thick material but does not make you look boxy it fits so nicely and is flattering the design is just gorgeous if you are considering this sweater get it you wont regret it

im normally a small 46 and i ordered a small it fits true to size the fit is perfect so if you want it slightly more relaxed order one size up



'Stunning This sweater is so beautiful on it is thick material but does not make you look boxy it fits so nicely and is flattering the design is just gorgeous if you are considering this sweater get it you wont regret it\n\nim normally a small 46 and i ordered a small it fits true to size the fit is perfect so if you want it slightly more relaxed order one size up'

In [101]:
text = show_step(prep.remove_numbers, text)

Paso: remove_numbers

Entrada:
stunning! this sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

Salida:
stunning! this sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (-) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.



In [102]:
text = show_step(prep.remove_punctuation, text)

Paso: remove_punctuation

Entrada:
stunning! this sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you will not regret it!

i am normally a small (-) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.

Salida:
stunning this sweater is so beautiful on it is thick material but does not make you look boxy it fits so nicely and is flattering the design is just gorgeous if you are considering this sweater get it you will not regret it

i am normally a small  and i ordered a small it fits true to size the fit is perfect so if you want it slightly more relaxed order one size up



Hasta este punto se realizaron las transformaciones básicas orientadas a estandarizar el texto, eliminando diferencias de formato que no aportan información semántica. El resultado es un texto limpio y consistente, listo para aplicar técnicas de procesamiento lingüístico como la tokenización, la eliminación de palabras vacías y la lematización.

In [103]:
tokens = show_step(prep.tokenize, text)

Paso: tokenize

Entrada:
stunning this sweater is so beautiful on it is thick material but does not make you look boxy it fits so nicely and is flattering the design is just gorgeous if you are considering this sweater get it you will not regret it

i am normally a small  and i ordered a small it fits true to size the fit is perfect so if you want it slightly more relaxed order one size up

Salida:
['stunning', 'this', 'sweater', 'is', 'so', 'beautiful', 'on', 'it', 'is', 'thick', 'material', 'but', 'does', 'not', 'make', 'you', 'look', 'boxy', 'it', 'fits', 'so', 'nicely', 'and', 'is', 'flattering', 'the', 'design', 'is', 'just', 'gorgeous', 'if', 'you', 'are', 'considering', 'this', 'sweater', 'get', 'it', 'you', 'will', 'not', 'regret', 'it', 'i', 'am', 'normally', 'a', 'small', 'and', 'i', 'ordered', 'a', 'small', 'it', 'fits', 'true', 'to', 'size', 'the', 'fit', 'is', 'perfect', 'so', 'if', 'you', 'want', 'it', 'slightly', 'more', 'relaxed', 'order', 'one', 'size', 'up']



In [104]:
tokens = show_step(prep.remove_stopwords, tokens)

Paso: remove_stopwords

Entrada:
['stunning', 'this', 'sweater', 'is', 'so', 'beautiful', 'on', 'it', 'is', 'thick', 'material', 'but', 'does', 'not', 'make', 'you', 'look', 'boxy', 'it', 'fits', 'so', 'nicely', 'and', 'is', 'flattering', 'the', 'design', 'is', 'just', 'gorgeous', 'if', 'you', 'are', 'considering', 'this', 'sweater', 'get', 'it', 'you', 'will', 'not', 'regret', 'it', 'i', 'am', 'normally', 'a', 'small', 'and', 'i', 'ordered', 'a', 'small', 'it', 'fits', 'true', 'to', 'size', 'the', 'fit', 'is', 'perfect', 'so', 'if', 'you', 'want', 'it', 'slightly', 'more', 'relaxed', 'order', 'one', 'size', 'up']

Salida:
['stunning', 'sweater', 'beautiful', 'thick', 'material', 'make', 'look', 'boxy', 'fits', 'nicely', 'flattering', 'design', 'gorgeous', 'considering', 'sweater', 'get', 'regret', 'normally', 'small', 'ordered', 'small', 'fits', 'true', 'size', 'fit', 'perfect', 'want', 'slightly', 'relaxed', 'order', 'one', 'size']



In [105]:
tokens = show_step(prep.lemmatize, tokens)

Paso: lemmatize

Entrada:
['stunning', 'sweater', 'beautiful', 'thick', 'material', 'make', 'look', 'boxy', 'fits', 'nicely', 'flattering', 'design', 'gorgeous', 'considering', 'sweater', 'get', 'regret', 'normally', 'small', 'ordered', 'small', 'fits', 'true', 'size', 'fit', 'perfect', 'want', 'slightly', 'relaxed', 'order', 'one', 'size']

Salida:
['stunning', 'sweater', 'beautiful', 'thick', 'material', 'make', 'look', 'boxy', 'fit', 'nicely', 'flattering', 'design', 'gorgeous', 'considering', 'sweater', 'get', 'regret', 'normally', 'small', 'ordered', 'small', 'fit', 'true', 'size', 'fit', 'perfect', 'want', 'slightly', 'relaxed', 'order', 'one', 'size']



In [106]:
text = show_step(prep.join_tokens, tokens)

Paso: join_tokens

Entrada:
['stunning', 'sweater', 'beautiful', 'thick', 'material', 'make', 'look', 'boxy', 'fit', 'nicely', 'flattering', 'design', 'gorgeous', 'considering', 'sweater', 'get', 'regret', 'normally', 'small', 'ordered', 'small', 'fit', 'true', 'size', 'fit', 'perfect', 'want', 'slightly', 'relaxed', 'order', 'one', 'size']

Salida:
stunning sweater beautiful thick material make look boxy fit nicely flattering design gorgeous considering sweater get regret normally small ordered small fit true size fit perfect want slightly relaxed order one size



### Conclusiones

El texto fue sometido a una secuencia de transformaciones destinada a reducir el ruido y estandarizar su representación antes de la extracción de características. El pipeline implementado incluye normalización, expansión de contracciones, tokenización, eliminación de palabras vacías, lematización y reconstrucción del texto.

Todas estas operaciones fueron encapsuladas en funciones reutilizables dentro del módulo src/data/preprocessing.py, permitiendo mantener el notebook como un registro del proceso y facilitando la reutilización del pipeline durante el entrenamiento y la evaluación del modelo.

## Aplicación del pipeline al conjunto de datos

Una vez verificadas individualmente las transformaciones del pipeline, se procede a aplicarlas sobre la totalidad de las reseñas.

El objetivo es obtener una representación textual consistente y lista para la etapa de extracción de características mediante TF-IDF. Para mantener una arquitectura modular, todo el proceso se encuentra encapsulado en una única función dentro del módulo `src.data.preprocessing`.

In [107]:
import importlib
import src.data.preprocessing as prep
importlib.reload(prep)

<module 'src.data.preprocessing' from 'c:\\Users\\lauta\\Desktop\\DataScience\\amazon_reviews_nlp\\src\\data\\preprocessing.py'>

In [108]:
df_raw = df.copy()

df["Text"] = df["Text"].apply(prep.preprocess_text)

In [109]:
comparison = pd.DataFrame({
    "Original": df_raw["Text"].head(3),
    "Procesado": df["Text"].head(3)
})

display(comparison)

,Original,Procesado
0,Absolutely wonderful - silky and sexy and comfortable,absolutely wonderful - silky sexy comfortable
1,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i ...","love dress ! sooo pretty . happened find store , glad never would ordered online petite . bought..."
2,Some major design flaws I had such high hopes for this dress and really wanted it to work for me...,major design flaw high hope dress really wanted work . initially ordered petite small ( usual si...


In [111]:
from pathlib import Path

output_path = Path("../data/processed/amazon_reviews_preprocessed.csv")

df.to_csv(output_path, index=False)

print("✓ Dataset preprocesado guardado correctamente.")
print(f"Ruta: {output_path}")

✓ Dataset preprocesado guardado correctamente.
Ruta: ..\data\processed\amazon_reviews_preprocessed.csv


In [112]:
display(df.head())

,Text,Rating,Age,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,absolutely wonderful - silky sexy comfortable,4,33,1,0,Initmates,Intimate,Intimates
1,"love dress ! sooo pretty . happened find store , glad never would ordered online petite . bought...",5,34,1,4,General,Dresses,Dresses
2,major design flaw high hope dress really wanted work . initially ordered petite small ( usual si...,3,60,0,0,General,Dresses,Dresses
3,"favorite buy ! love , love , love jumpsuit . fun , flirty , fabulous ! every time wear , get not...",5,50,1,0,General Petite,Bottoms,Pants
4,flattering shirt shirt flattering due adjustable front tie . perfect length wear legging sleevel...,5,47,1,6,General,Tops,Blouses


## Conclusiones

En esta etapa se realizó el preprocesamiento completo de las reseñas textuales mediante un pipeline modular implementado en `src.data.preprocessing`.

Las transformaciones aplicadas permitieron reducir el ruido del texto, estandarizar la representación lingüística y preparar las reseñas para la extracción de características basada en TF-IDF.

Como resultado, se obtuvo un conjunto de datos limpio y consistente, almacenado en la carpeta `data/processed`, que será utilizado como entrada para la etapa de modelado y entrenamiento de los clasificadores.